# Download a YouTube video as MP3

#### Install dependencies

In [ ]:
# `yt-dlp` via pip
%pip install yt-dlp

In [ ]:
# `ffmpeg` via apt / static build
import shutil
import subprocess
import sys
from pathlib import Path

if shutil.which("ffmpeg"):
    print(f"ffmpeg already available: {shutil.which('ffmpeg')}")
elif sys.platform.startswith("linux") and shutil.which("apt-get"):
    subprocess.run(["sudo", "apt-get", "update"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "-y", "ffmpeg"], check=True)
else:
    local_bin = Path.home() / ".local" / "bin"
    local_bin.mkdir(parents=True, exist_ok=True)
    import tarfile
    import urllib.request

    url = "https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz"
    archive = Path.cwd() / "ffmpeg-release-amd64-static.tar.xz"
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, "r:xz") as tar:
        tar.extractall(Path.cwd())
    root = next(p for p in Path.cwd().iterdir() if p.is_dir() and p.name.startswith("ffmpeg"))
    for tool in ("ffmpeg", "ffprobe"):
        (root / tool).rename(local_bin / tool)

assert shutil.which("ffmpeg"), "ffmpeg still not found on PATH"
print("ffmpeg is installed:", shutil.which("ffmpeg"))

#### Function definitions

In [ ]:
import json
import re
import threading
import uuid
from pathlib import Path

from yt_dlp import YoutubeDL

_CACHE_LOCK = threading.Lock()

def download_from_youtube_as_mp3(url: str) -> tuple[bool, Path | None]:
    if not re.match(r"(https?://)?(www\.)?(youtube\.com|youtu\.be)/", url):
        raise ValueError("The provided URL is not a valid YouTube video URL.")

    cache_file = Path.cwd().resolve() / "download_cache.json"
    cache = {}
    if cache_file.exists():
        try:
            with open(cache_file, "r") as f:
                cache = json.load(f)
        except (json.JSONDecodeError, OSError):
            cache = {}
        if url in cache:
            cached_path = Path(cache[url])
            if cached_path.exists():
                print("Using cached download.")
                return True, cached_path

    output_folder = Path.cwd().resolve() / "downloads"
    output_folder.mkdir(exist_ok=True)

    temp_name = str(uuid.uuid4())
    temp_path = str(output_folder / f"{temp_name}.%(ext)s")

    opts = {
        "format": "bestaudio/best",
        "extractaudio": True,
        "audioformat": "mp3",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3"}],
        "outtmpl": temp_path,
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }

    try:
        with YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)

            if info is None:
                print("Failed to download video.")
                return False, None

            title = re.sub(r'[<>:"/\\|?*]', " ", info.get("title", temp_name))
            final_path = output_folder / f"{title}.mp3"
            downloaded_path = output_folder / f"{temp_name}.mp3"

            if downloaded_path.exists():
                downloaded_path.rename(final_path)

            with _CACHE_LOCK:
                cache = {}
                if cache_file.exists():
                    try:
                        cache = json.loads(cache_file.read_text())
                    except (json.JSONDecodeError, OSError):
                        cache = {}
                cache[url] = str(final_path)
                cache_file.write_text(json.dumps(cache, indent=2))

            return True, final_path

    except Exception as e:
        print(f"Error: {e}")
        return False, None

#### Usage example

In [ ]:
ok, path = download_from_youtube_as_mp3(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)
print(ok, path)